# Set-up  

In [4]:
from google.colab import drive, userdata
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google import genai
from google.genai import types
import time

import pandas as pd

# Gemini Client Initiation

In [31]:
client = genai.Client(api_key=userdata.get('gemini_api_key'))

# File name will be visible in citations
file_search_store = client.file_search_stores.create(
    config={'display_name': 'bot_policy_20240702'
    })

# Add File To File Search Store

In [47]:
def store_file(cilent, file_name, file_path='/content/drive/MyDrive/KKP/'):
  full_file_path = f"{file_path}{file_name}"

  operation = client.file_search_stores.upload_to_file_search_store(
    file=full_file_path,
    file_search_store_name=file_search_store.name,
    config={
        'display_name' : file_name,
    }
  )
  return operation

In [49]:
# file_name = '20240702_BOT_Policy.pdf'
file_name = 'Analysis_of_Legal1.pdf'

operation = store_file(client, file_name)

while not operation.done:
    time.sleep(5)
    operation = client.operations.get(operation)


In [33]:
# List all file search stores
for fs in client.file_search_stores.list():
    print(f" {file_search_store.name} - {file_search_store.display_name}")

 fileSearchStores/botpolicy20240702-w2uusfol5kgo - bot_policy_20240702
 fileSearchStores/botpolicy20240702-w2uusfol5kgo - bot_policy_20240702


In [ ]:
# client.file_search_stores.delete(name='fileSearchStores/botpolicy20240702-2ot0jq8eztge', config={'force':True})

# Gemini Test Prompt

In [41]:
def ask_legal_gemini(question):
  response = client.models.generate_content(
      model="gemini-2.5-flash",
            # contents=f"""คุณเป็นผู้เชี่ยวชาญทางกฎหมายของธนาคารเอกชนแห่งหนึ่ง และตอบคำถามต่อไปนี้อย่างรอบคอบและกระชับ ไม่ยาวจนเกินไป\n{question}""",
      contents=f"""{question}""",
      config=types.GenerateContentConfig(
          tools=[
              types.Tool(
                  file_search=types.FileSearch(
                      file_search_store_names=[file_search_store.name]
                  )
              )
          ]
      )
  )

  return response

In [42]:
question="""1. ที่ดินที่ได้มาจากการที่บิดายกให้บุตรโดยเสน่หา บิดาซึ่งเป็นผู้ให้ยังสามารถเรียกคืนจากบุตรได้หรือไม่ เรียกคืนจากสาเหตุใดได้บ้าง
2. กรณีที่บุตรนำที่ดินไปดำเนินการจัดสรรแล้ว ยังจะสามารถถูกเรียกคืนการให้จากผู้ให้ได้หรือไม่"""

print(ask_legal_gemini(question).text)

ผู้ให้ยังสามารถเรียกคืนการให้ที่ดินที่ได้มอบให้บุตรโดยเสน่หาได้ แต่ต้องเป็นไปตามเงื่อนไขที่กฎหมายกำหนดไว้. การเรียกคืนการให้ไม่ได้ทำได้โดยง่าย และต้องมีเหตุผลตามกฎหมายเท่านั้น โดยทั่วไปแล้ว เหตุผลหลักที่ผู้ให้จะสามารถเรียกคืนการให้ได้แก่:

1.  **ประพฤติเนรคุณ:** หากผู้รับการให้ (บุตร) กระทำการอันเป็นประพฤติเนรคุณอย่างร้ายแรงต่อผู้ให้ (บิดา) เช่น ทำร้ายร่างกาย, หมิ่นประมาทอย่างร้ายแรง, หรือไม่ให้การอุปการะเลี้ยงดูที่จำเป็นแก่ผู้ให้ในยามยากไร้ โดยที่ไม่ได้รับการช่วยเหลือจากผู้รับการให้.
2.  **ผู้ให้ยากไร้ลง:** หากผู้ให้ตกเป็นผู้ยากไร้ลงภายหลังการให้ และไม่มีทรัพย์สินเพียงพอเลี้ยงชีพ หรือต้องเลี้ยงดูบุคคลอื่นตามหน้าที่ตามกฎหมาย ผู้ให้สามารถเรียกคืนการให้ได้เท่าที่จำเป็นเพื่อการเลี้ยงชีพของตนและบุคคลที่ต้องอุปการะ.
3.  **การไม่ปฏิบัติตามเงื่อนไข:** หากการให้มีเงื่อนไขผูกพันและผู้รับการให้ไม่ปฏิบัติตามเงื่อนไขนั้น ผู้ให้ก็อาจมีสิทธิเรียกคืนการให้ได้.

**กรณีที่บุตรนำที่ดินไปดำเนินการจัดสรรแล้ว ยังจะสามารถถูกเรียกคืนการให้จากผู้ให้ได้หรือไม่:**

หากบุตรได้นำที่ดินไปดำเนินการจัดสรรแล้ว การเรี

# Import Test Cases From File

In [37]:
test_df = pd.read_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย (ปรับเพิ่มสำหรับใช้งาน AI)_Internal Use_Extract.xlsx",
                        sheet_name="Legal Analysis to BU",
                        skiprows=[0,1])

test_df.columns = [x.strip() for x in test_df.columns]
test_df = test_df.dropna(subset=["ข้อเท็จจริง/คำถาม"])
test_df

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,วันที่ออกความเห็น,Project/Client (กรณีบุคคลธรรมดา ให้ระบุชื่อสกุลบางส่วน เช่น สมxxx โชคxxx),BU,ประเด็นทางกฎหมาย / Key words,ข้อเท็จจริง/คำถาม,สรุปความเห็นทางกฎหมาย,รายละเอียดความเห็นทางกฎหมายแบบที่ส่งให้ BU,กฎหมายที่เกี่ยวข้อง,ฎีกาที่เกี่ยวข้อง,ผู้จัดทำความเห็น
1,2023-04-30 00:00:00,ABCD,CBG,โปรดระบุประเด็นทางกฎหมายของความเห็น เช่น ผู้ให...,โปรดระบุคำถาม/ข้อเท็จจริงที่ได้รับ\nเช่น ผู้ให...,โปรดปรับ/เลือกใช้คำให้อยู่ในรูปแบบธงคำตอบ ที่ส...,โปรดระบุรายละเอียดความเห็นทางกฎหมายแบบที่ส่งให...,โปรดระบุชื่อกฎหมาย (มาตราที่เกี่ยวข้อง),โปรดระบุเลขคำพิพากษาฎีกา (หากมี),ชื่อจริง\n
2,17-Mar-23,NaN,CL-RE,ผู้ให้หลักประกันบัญชีเงินฝากซึ่งจด BSA ไว้เสีย...,ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเส...,ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาค...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1599 และ 160...,NaN,NaN
3,2023-01-26 00:00:00,NaN,CL-RE,สิทธิตาม BSA ในเงินฝากกับบุริมสิทธิเจ้าหนี้ค่า...,ธนาคารต้องส่งมอบเงินฝากซึ่งจด BSA ไว้ ให้แก่เจ...,กรณีที่มีบุริมสิทธิแย้งกับสิทธิตามสัญญาหลักประ...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 251, 253(3),...",NaN,NaN
4,2023-02-09 00:00:00,NaN,CL-RE,ข้อยกเว้นที่ผู้จำนองเข้าทำสัญญาค้ำประกันได้,ผู้จำนองที่เป็นผู้ถือหุ้นผู้กู้ เข้าทำสัญญาค้ำ...,ข้อตกลงใดอันมีผลให้ผู้จำนองรับรับผิดอย่างผู้ค้...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 727/1),NaN,NaN
5,20-Jan-23,NaN,CL-RE,ผู้จัดการมรดกตามพินัยกรรม\nความยินยอมของทายาท\n,ผู้กู้ซื้อที่ดินจากผู้จัดการมรดกตามพินัยกรรม แ...,ผู้จัดการมรดกสามารถตั้งขึ้นโดยคำสั่งศาลหรือพิน...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1658, มาตรา ...",คำพิพากษาศาลฎีกาที่ 814/2554,NaN
6,2023-02-08 00:00:00,NaN,CL-RE,คู่สมรสของผู้ค้ำประกันไม่ได้ลงนามให้ความยินยอม\n,คู่สมรสของผู้ค้ำประกันไม่ได้ลงนามให้ความยินยอม...,การที่คู่สมรสของผู้ค้ำประกันไม่ได้ลงนามให้ความ...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1472, 1474, ...",คำพิพากษาศาลฎีกาที่ 4650/2545,NaN
7,2023-02-07 00:00:00,NaN,CL-GI,ความรับผิดในฐานะผู้ค้ำประกันตามระยะเวลาที่ขยาย...,ผู้กู้ขอให้ออกหนังสือค้ำประกัน/LG ฉบับใหม่ หลั...,การที่หนังสือค้ำประกัน/LG กำหนดให้ธนาคารให้ควา...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 700),NaN,NaN
8,2023-02-13 00:00:00,NaN,CL-RE,การก่อสร้างระหว่างยื่นคำขอต่ออายุใบอนุญาตก่อสร...,หากผู้กู้ได้นำส่งใบอนุญาตก่อสร้างให้แก่ธนาคารต...,โดยหลักกฎหมาย ใบอนุญาตก่อสร้างสามารถใช้ได้ตามร...,NaN,"พ.ร.บ.ควบคุมอาคาร พ.ศ. 2522 (มาตรา 21, 35 และ ...",NaN,NaN
9,2023-02-13 00:00:00,NaN,CL-GI,การ rollover ตั๋วสัญญาใช้เงิน (PN) ภายหลังผู้ค...,กรณี rollover ตั๋วสัญญาใช้เงิน (PN) ถือว่ายังเ...,หากวงเงิน PN ไม่มีข้อกำหนดให้ rollover ได้ หนี...,NaN,NaN,NaN,NaN
10,2023-03-10 00:00:00,NaN,CL-RE,ผู้ให้กู้ยกเลิกวงเงินสินเชื่อ โดยที่ผู้กู้ไม่ย...,เมื่อไม่มีภาระหนี้แล้ว ผู้ให้กู้ต้องการที่จะยก...,เมื่อผู้กู้ได้รับอนุมัติวงเงินสินเชื่อประเภท u...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 386),NaN,NaN


In [44]:
test_df['AI_RESPONSE'] = test_df["ข้อเท็จจริง/คำถาม"].apply(lambda x: ask_legal_gemini(x).text)

test_df.head(5)

,วันที่ออกความเห็น,Project/Client (กรณีบุคคลธรรมดา ให้ระบุชื่อสกุลบางส่วน เช่น สมxxx โชคxxx),BU,ประเด็นทางกฎหมาย / Key words,ข้อเท็จจริง/คำถาม,สรุปความเห็นทางกฎหมาย,รายละเอียดความเห็นทางกฎหมายแบบที่ส่งให้ BU,กฎหมายที่เกี่ยวข้อง,ฎีกาที่เกี่ยวข้อง,ผู้จัดทำความเห็น,AI_RESPONSE
1,2023-04-30 00:00:00,ABCD,CBG,โปรดระบุประเด็นทางกฎหมายของความเห็น เช่น ผู้ให...,โปรดระบุคำถาม/ข้อเท็จจริงที่ได้รับ\nเช่น ผู้ให...,โปรดปรับ/เลือกใช้คำให้อยู่ในรูปแบบธงคำตอบ ที่ส...,โปรดระบุรายละเอียดความเห็นทางกฎหมายแบบที่ส่งให...,โปรดระบุชื่อกฎหมาย (มาตราที่เกี่ยวข้อง),โปรดระบุเลขคำพิพากษาฎีกา (หากมี),ชื่อจริง\n,เพื่อให้เห็นภาพของคำถาม/ข้อเท็จจริงที่อาจได้รั...
2,17-Mar-23,NaN,CL-RE,ผู้ให้หลักประกันบัญชีเงินฝากซึ่งจด BSA ไว้เสีย...,ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเส...,ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาค...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1599 และ 160...,NaN,NaN,จากข้อมูลที่ค้นพบ ไม่พบรายละเอียดที่ระบุโดยตรง...
3,2023-01-26 00:00:00,NaN,CL-RE,สิทธิตาม BSA ในเงินฝากกับบุริมสิทธิเจ้าหนี้ค่า...,ธนาคารต้องส่งมอบเงินฝากซึ่งจด BSA ไว้ ให้แก่เจ...,กรณีที่มีบุริมสิทธิแย้งกับสิทธิตามสัญญาหลักประ...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 251, 253(3),...",NaN,NaN,ธนาคารแห่งประเทศไทยอนุญาตให้ธนาคารพาณิชย์และบร...
4,2023-02-09 00:00:00,NaN,CL-RE,ข้อยกเว้นที่ผู้จำนองเข้าทำสัญญาค้ำประกันได้,ผู้จำนองที่เป็นผู้ถือหุ้นผู้กู้ เข้าทำสัญญาค้ำ...,ข้อตกลงใดอันมีผลให้ผู้จำนองรับรับผิดอย่างผู้ค้...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 727/1),NaN,NaN,จากข้อมูลที่ได้จากการค้นหา ไม่ได้ระบุโดยตรงว่า...
5,20-Jan-23,NaN,CL-RE,ผู้จัดการมรดกตามพินัยกรรม\nความยินยอมของทายาท\n,ผู้กู้ซื้อที่ดินจากผู้จัดการมรดกตามพินัยกรรม แ...,ผู้จัดการมรดกสามารถตั้งขึ้นโดยคำสั่งศาลหรือพิน...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1658, มาตรา ...",คำพิพากษาศาลฎีกาที่ 814/2554,NaN,ผู้กู้ที่ซื้อที่ดินจากผู้จัดการมรดกตามพินัยกรร...


In [45]:
test_df.to_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย AI_RESPONSES.xlsx")